# Assignment 4
## Description
In this assignment you must read in a file of metropolitan regions and associated sports teams from [assets/wikipedia_data.html](assets/wikipedia_data.html) and answer some questions about each metropolitan region. Each of these regions may have one or more teams from the "Big 4": NFL (football, in [assets/nfl.csv](assets/nfl.csv)), MLB (baseball, in [assets/mlb.csv](assets/mlb.csv)), NBA (basketball, in [assets/nba.csv](assets/nba.csv) or NHL (hockey, in [assets/nhl.csv](assets/nhl.csv)). Please keep in mind that all questions are from the perspective of the metropolitan region, and that this file is the "source of authority" for the location of a given sports team. Thus teams which are commonly known by a different area (e.g. "Oakland Raiders") need to be mapped into the metropolitan region given (e.g. San Francisco Bay Area). This will require some human data understanding outside of the data you've been given (e.g. you will have to hand-code some names, and might need to google to find out where teams are)!

For each sport I would like you to answer the question: **what is the win/loss ratio's correlation with the population of the city it is in?** Win/Loss ratio refers to the number of wins over the number of wins plus the number of losses. Remember that to calculate the correlation with [`pearsonr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html), so you are going to send in two ordered lists of values, the populations from the wikipedia_data.html file and the win/loss ratio for a given sport in the same order. Average the win/loss ratios for those cities which have multiple teams of a single sport. Each sport is worth an equal amount in this assignment (20%\*4=80%) of the grade for this assignment. You should only use data **from year 2018** for your analysis -- this is important!

## Notes

1. Do not include data about the MLS or CFL in any of the work you are doing, we're only interested in the Big 4 in this assignment.
2. I highly suggest that you first tackle the four correlation questions in order, as they are all similar and worth the majority of grades for this assignment. This is by design!
3. It's fair game to talk with peers about high level strategy as well as the relationship between metropolitan areas and sports teams. However, do not post code solving aspects of the assignment (including such as dictionaries mapping areas to teams, or regexes which will clean up names).
4. There may be more teams than the assert statements test, remember to collapse multiple teams in one city into a single value!

As this assignment utilizes global variables in the skeleton code, to avoid having errors in your code you can either:

1. You can place all of your code within the function definitions for all of the questions (other than import statements).
2. You can create copies of all the global variables with the copy() method and proceed as usual.

## Question 1
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NHL** using **2018** data.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPERS  (used by all 5 questions)
# ─────────────────────────────────────────────────────────────────────────────

# Nicknames that are two words — must NOT be split when tokenising city entries
TWO_WORD_NICKNAMES = [
    'Red Sox', 'White Sox', 'Blue Jays', 'Red Wings',
    'Maple Leafs', 'Blue Jackets', 'Golden Knights', 'Trail Blazers'
]

def get_team_key(raw_name):
    """
    Extract a team's matchable nickname from its full name in the CSV.
    • Strips footnote markers (* + ) and NBA seed annotations (\\xa0(N)).
    • Returns last TWO words for two-word nicknames, otherwise last word.
    Examples:
        'Boston Red Sox*'            → 'Red Sox'
        'Portland Trail Blazers* (3)' → 'Trail Blazers'
        'Dallas Stars'               → 'Stars'
    """
    name = re.sub(r'[\*\+]', '', raw_name)
    name = re.sub(r'\xa0.*', '', name).strip()   # remove non-breaking space + seed
    words = name.split()
    if len(words) >= 2 and ' '.join(words[-2:]) in TWO_WORD_NICKNAMES:
        return ' '.join(words[-2:])
    return words[-1]


def tokenize_city_entry(raw):
    """
    Split a Wikipedia city cell (e.g. 'Cubs White Sox') into individual
    team nicknames while preserving two-word nicknames as single tokens.
    Returns a list of nickname strings (empty list for '—' / blank cells).
    """
    s = re.sub(r'\[.*?\]', '', raw).strip()   # strip footnotes like [note 3]
    if not s or s == '—':
        return []
    # Temporarily replace spaces inside two-word nicknames so split() won't break them
    protected = s
    for tw in sorted(TWO_WORD_NICKNAMES, key=len, reverse=True):
        protected = protected.replace(tw, tw.replace(' ', '\x00'))
    tokens = protected.split()
    return [t.replace('\x00', ' ') for t in tokens]


def load_cities(sport):
    """
    Load wikipedia_data.html, keep only Metropolitan area + population +
    the requested sport column, then explode multi-team cells into one row
    per team nickname.
    """
    cities = pd.read_html('assets/wikipedia_data.html')[1]
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities['Population (2016 est.)[8]'] = pd.to_numeric(
        cities['Population (2016 est.)[8]'], errors='coerce')
    cities[sport] = cities[sport].apply(tokenize_city_entry)
    cities = cities.explode(sport)
    # drop rows with no team (blank or dash)
    cities = cities[
        cities[sport].str.strip().ne('') & cities[sport].str.strip().ne('—')
    ].copy()
    return cities


def build_sport_df(sport, csv_file, wl_col=None):
    """
    Generic builder used by Q1-Q5.
    • Loads the sport CSV, keeps only 2018 rows, drops division-header rows.
    • Merges with city population data on team nickname.
    • Averages W-L% for cities with multiple teams.
    • Returns a DataFrame with columns: Metropolitan area | Population | WL
    """
    raw = pd.read_csv(f'assets/{csv_file}')
    cities = load_cities(sport)

    # Keep only 2018 and real team rows (filter out division-header rows)
    raw = raw[raw['year'] == 2018].copy()
    raw = raw[pd.to_numeric(raw['W'], errors='coerce').notna()].copy()

    # Build the matchable key
    raw['team_key'] = raw['team'].apply(get_team_key)

    # Win-loss percentage
    if wl_col:
        raw['WL'] = pd.to_numeric(raw[wl_col], errors='coerce')
    else:                                       # NHL has no pre-computed W-L%
        raw['W'] = raw['W'].astype(int)
        raw['L'] = raw['L'].astype(int)
        raw['WL'] = raw['W'] / (raw['W'] + raw['L'])

    # Merge and average multi-team cities
    df = pd.merge(cities, raw, left_on=sport, right_on='team_key')
    df = df.groupby('Metropolitan area', as_index=False).agg(
        Population=('Population (2016 est.)[8]', 'first'),
        WL=('WL', 'mean')
    )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 1 — NHL  (expected: 28 metropolitan areas, r ≈ 0.0125)
# ─────────────────────────────────────────────────────────────────────────────

def nhl_correlation():
    df = build_sport_df('NHL', 'nhl.csv')          # no wl_col → computed from W/L

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q1: There should be 28 teams being analysed for NHL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 2 — NBA  (expected: 28 metropolitan areas, r ≈ -0.1764)
# ─────────────────────────────────────────────────────────────────────────────

def nba_correlation():
    df = build_sport_df('NBA', 'nba.csv', wl_col='W/L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q2: There should be 28 teams being analysed for NBA"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 3 — MLB  (expected: 26 metropolitan areas, r ≈ 0.1500)
# ─────────────────────────────────────────────────────────────────────────────

def mlb_correlation():
    df = build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, \
        "Q3: There should be 26 teams being analysed for MLB"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 4 — NFL  (expected: 29 metropolitan areas, r ≈ 0.0043)
# ─────────────────────────────────────────────────────────────────────────────

def nfl_correlation():
    df = build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, \
        "Q4: There should be 29 teams being analysed for NFL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 5 — Paired t-tests across all sport pairs
# ─────────────────────────────────────────────────────────────────────────────

def sports_team_performance():
    """
    For every pair of sports, run a paired t-test (ttest_rel) on the W-L%
    of cities that have teams in BOTH sports.  Returns a 4×4 DataFrame of
    p-values; the diagonal is NaN (same sport vs itself).

    Key assertions (checked by the grader):
        NBA vs NHL  p ≈ 0.022   (can reject null — they perform differently)
        MLB vs NFL  p ≈ 0.803   (cannot reject null)
    """
    # Reuse the same cleaned DataFrames from Q1-Q4
    leagues = {
        'NFL': build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
        'NBA': build_sport_df('NBA', 'nba.csv', wl_col='W/L%')[['Metropolitan area', 'WL']],
        'NHL': build_sport_df('NHL', 'nhl.csv')[['Metropolitan area', 'WL']],
        'MLB': build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
    }

    names = list(leagues.keys())
    p_values = pd.DataFrame(index=names, columns=names, dtype=float)

    for l1 in names:
        for l2 in names:
            if l1 == l2:
                p_values.loc[l1, l2] = np.nan
            else:
                # Only cities present in BOTH leagues
                merged = pd.merge(leagues[l1], leagues[l2], on='Metropolitan area')
                _, p = stats.ttest_rel(merged['WL_x'], merged['WL_y'])
                p_values.loc[l1, l2] = p

    assert abs(p_values.loc['NBA', 'NHL'] - 0.02) <= 1e-2, \
        "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc['MLB', 'NFL'] - 0.80) <= 1e-2, \
        "The MLB-NFL p-value should be around 0.80"

    return p_values


# ─────────────────────────────────────────────────────────────────────────────
# Quick sanity-check when run directly
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print(f"Q1 NHL  correlation : {nhl_correlation():.6f}")
    print(f"Q2 NBA  correlation : {nba_correlation():.6f}")
    print(f"Q3 MLB  correlation : {mlb_correlation():.6f}")
    print(f"Q4 NFL  correlation : {nfl_correlation():.6f}")
    print()
    print("Q5 p-values:")
    print(sports_team_performance().to_string())

## Question 2
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NBA** using **2018** data.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPERS  (used by all 5 questions)
# ─────────────────────────────────────────────────────────────────────────────

# Nicknames that are two words — must NOT be split when tokenising city entries
TWO_WORD_NICKNAMES = [
    'Red Sox', 'White Sox', 'Blue Jays', 'Red Wings',
    'Maple Leafs', 'Blue Jackets', 'Golden Knights', 'Trail Blazers'
]

def get_team_key(raw_name):
    """
    Extract a team's matchable nickname from its full name in the CSV.
    • Strips footnote markers (* + ) and NBA seed annotations (\\xa0(N)).
    • Returns last TWO words for two-word nicknames, otherwise last word.
    Examples:
        'Boston Red Sox*'            → 'Red Sox'
        'Portland Trail Blazers* (3)' → 'Trail Blazers'
        'Dallas Stars'               → 'Stars'
    """
    name = re.sub(r'[\*\+]', '', raw_name)
    name = re.sub(r'\xa0.*', '', name).strip()   # remove non-breaking space + seed
    words = name.split()
    if len(words) >= 2 and ' '.join(words[-2:]) in TWO_WORD_NICKNAMES:
        return ' '.join(words[-2:])
    return words[-1]


def tokenize_city_entry(raw):
    """
    Split a Wikipedia city cell (e.g. 'Cubs White Sox') into individual
    team nicknames while preserving two-word nicknames as single tokens.
    Returns a list of nickname strings (empty list for '—' / blank cells).
    """
    s = re.sub(r'\[.*?\]', '', raw).strip()   # strip footnotes like [note 3]
    if not s or s == '—':
        return []
    # Temporarily replace spaces inside two-word nicknames so split() won't break them
    protected = s
    for tw in sorted(TWO_WORD_NICKNAMES, key=len, reverse=True):
        protected = protected.replace(tw, tw.replace(' ', '\x00'))
    tokens = protected.split()
    return [t.replace('\x00', ' ') for t in tokens]


def load_cities(sport):
    """
    Load wikipedia_data.html, keep only Metropolitan area + population +
    the requested sport column, then explode multi-team cells into one row
    per team nickname.
    """
    cities = pd.read_html('assets/wikipedia_data.html')[1]
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities['Population (2016 est.)[8]'] = pd.to_numeric(
        cities['Population (2016 est.)[8]'], errors='coerce')
    cities[sport] = cities[sport].apply(tokenize_city_entry)
    cities = cities.explode(sport)
    # drop rows with no team (blank or dash)
    cities = cities[
        cities[sport].str.strip().ne('') & cities[sport].str.strip().ne('—')
    ].copy()
    return cities


def build_sport_df(sport, csv_file, wl_col=None):
    """
    Generic builder used by Q1-Q5.
    • Loads the sport CSV, keeps only 2018 rows, drops division-header rows.
    • Merges with city population data on team nickname.
    • Averages W-L% for cities with multiple teams.
    • Returns a DataFrame with columns: Metropolitan area | Population | WL
    """
    raw = pd.read_csv(f'assets/{csv_file}')
    cities = load_cities(sport)

    # Keep only 2018 and real team rows (filter out division-header rows)
    raw = raw[raw['year'] == 2018].copy()
    raw = raw[pd.to_numeric(raw['W'], errors='coerce').notna()].copy()

    # Build the matchable key
    raw['team_key'] = raw['team'].apply(get_team_key)

    # Win-loss percentage
    if wl_col:
        raw['WL'] = pd.to_numeric(raw[wl_col], errors='coerce')
    else:                                       # NHL has no pre-computed W-L%
        raw['W'] = raw['W'].astype(int)
        raw['L'] = raw['L'].astype(int)
        raw['WL'] = raw['W'] / (raw['W'] + raw['L'])

    # Merge and average multi-team cities
    df = pd.merge(cities, raw, left_on=sport, right_on='team_key')
    df = df.groupby('Metropolitan area', as_index=False).agg(
        Population=('Population (2016 est.)[8]', 'first'),
        WL=('WL', 'mean')
    )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 1 — NHL  (expected: 28 metropolitan areas, r ≈ 0.0125)
# ─────────────────────────────────────────────────────────────────────────────

def nhl_correlation():
    df = build_sport_df('NHL', 'nhl.csv')          # no wl_col → computed from W/L

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q1: There should be 28 teams being analysed for NHL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 2 — NBA  (expected: 28 metropolitan areas, r ≈ -0.1764)
# ─────────────────────────────────────────────────────────────────────────────

def nba_correlation():
    df = build_sport_df('NBA', 'nba.csv', wl_col='W/L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q2: There should be 28 teams being analysed for NBA"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 3 — MLB  (expected: 26 metropolitan areas, r ≈ 0.1500)
# ─────────────────────────────────────────────────────────────────────────────

def mlb_correlation():
    df = build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, \
        "Q3: There should be 26 teams being analysed for MLB"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 4 — NFL  (expected: 29 metropolitan areas, r ≈ 0.0043)
# ─────────────────────────────────────────────────────────────────────────────

def nfl_correlation():
    df = build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, \
        "Q4: There should be 29 teams being analysed for NFL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 5 — Paired t-tests across all sport pairs
# ─────────────────────────────────────────────────────────────────────────────

def sports_team_performance():
    """
    For every pair of sports, run a paired t-test (ttest_rel) on the W-L%
    of cities that have teams in BOTH sports.  Returns a 4×4 DataFrame of
    p-values; the diagonal is NaN (same sport vs itself).

    Key assertions (checked by the grader):
        NBA vs NHL  p ≈ 0.022   (can reject null — they perform differently)
        MLB vs NFL  p ≈ 0.803   (cannot reject null)
    """
    # Reuse the same cleaned DataFrames from Q1-Q4
    leagues = {
        'NFL': build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
        'NBA': build_sport_df('NBA', 'nba.csv', wl_col='W/L%')[['Metropolitan area', 'WL']],
        'NHL': build_sport_df('NHL', 'nhl.csv')[['Metropolitan area', 'WL']],
        'MLB': build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
    }

    names = list(leagues.keys())
    p_values = pd.DataFrame(index=names, columns=names, dtype=float)

    for l1 in names:
        for l2 in names:
            if l1 == l2:
                p_values.loc[l1, l2] = np.nan
            else:
                # Only cities present in BOTH leagues
                merged = pd.merge(leagues[l1], leagues[l2], on='Metropolitan area')
                _, p = stats.ttest_rel(merged['WL_x'], merged['WL_y'])
                p_values.loc[l1, l2] = p

    assert abs(p_values.loc['NBA', 'NHL'] - 0.02) <= 1e-2, \
        "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc['MLB', 'NFL'] - 0.80) <= 1e-2, \
        "The MLB-NFL p-value should be around 0.80"

    return p_values


# ─────────────────────────────────────────────────────────────────────────────
# Quick sanity-check when run directly
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print(f"Q1 NHL  correlation : {nhl_correlation():.6f}")
    print(f"Q2 NBA  correlation : {nba_correlation():.6f}")
    print(f"Q3 MLB  correlation : {mlb_correlation():.6f}")
    print(f"Q4 NFL  correlation : {nfl_correlation():.6f}")
    print()
    print("Q5 p-values:")
    print(sports_team_performance().to_string())

## Question 3
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **MLB** using **2018** data.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPERS  (used by all 5 questions)
# ─────────────────────────────────────────────────────────────────────────────

# Nicknames that are two words — must NOT be split when tokenising city entries
TWO_WORD_NICKNAMES = [
    'Red Sox', 'White Sox', 'Blue Jays', 'Red Wings',
    'Maple Leafs', 'Blue Jackets', 'Golden Knights', 'Trail Blazers'
]

def get_team_key(raw_name):
    """
    Extract a team's matchable nickname from its full name in the CSV.
    • Strips footnote markers (* + ) and NBA seed annotations (\\xa0(N)).
    • Returns last TWO words for two-word nicknames, otherwise last word.
    Examples:
        'Boston Red Sox*'            → 'Red Sox'
        'Portland Trail Blazers* (3)' → 'Trail Blazers'
        'Dallas Stars'               → 'Stars'
    """
    name = re.sub(r'[\*\+]', '', raw_name)
    name = re.sub(r'\xa0.*', '', name).strip()   # remove non-breaking space + seed
    words = name.split()
    if len(words) >= 2 and ' '.join(words[-2:]) in TWO_WORD_NICKNAMES:
        return ' '.join(words[-2:])
    return words[-1]


def tokenize_city_entry(raw):
    """
    Split a Wikipedia city cell (e.g. 'Cubs White Sox') into individual
    team nicknames while preserving two-word nicknames as single tokens.
    Returns a list of nickname strings (empty list for '—' / blank cells).
    """
    s = re.sub(r'\[.*?\]', '', raw).strip()   # strip footnotes like [note 3]
    if not s or s == '—':
        return []
    # Temporarily replace spaces inside two-word nicknames so split() won't break them
    protected = s
    for tw in sorted(TWO_WORD_NICKNAMES, key=len, reverse=True):
        protected = protected.replace(tw, tw.replace(' ', '\x00'))
    tokens = protected.split()
    return [t.replace('\x00', ' ') for t in tokens]


def load_cities(sport):
    """
    Load wikipedia_data.html, keep only Metropolitan area + population +
    the requested sport column, then explode multi-team cells into one row
    per team nickname.
    """
    cities = pd.read_html('assets/wikipedia_data.html')[1]
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities['Population (2016 est.)[8]'] = pd.to_numeric(
        cities['Population (2016 est.)[8]'], errors='coerce')
    cities[sport] = cities[sport].apply(tokenize_city_entry)
    cities = cities.explode(sport)
    # drop rows with no team (blank or dash)
    cities = cities[
        cities[sport].str.strip().ne('') & cities[sport].str.strip().ne('—')
    ].copy()
    return cities


def build_sport_df(sport, csv_file, wl_col=None):
    """
    Generic builder used by Q1-Q5.
    • Loads the sport CSV, keeps only 2018 rows, drops division-header rows.
    • Merges with city population data on team nickname.
    • Averages W-L% for cities with multiple teams.
    • Returns a DataFrame with columns: Metropolitan area | Population | WL
    """
    raw = pd.read_csv(f'assets/{csv_file}')
    cities = load_cities(sport)

    # Keep only 2018 and real team rows (filter out division-header rows)
    raw = raw[raw['year'] == 2018].copy()
    raw = raw[pd.to_numeric(raw['W'], errors='coerce').notna()].copy()

    # Build the matchable key
    raw['team_key'] = raw['team'].apply(get_team_key)

    # Win-loss percentage
    if wl_col:
        raw['WL'] = pd.to_numeric(raw[wl_col], errors='coerce')
    else:                                       # NHL has no pre-computed W-L%
        raw['W'] = raw['W'].astype(int)
        raw['L'] = raw['L'].astype(int)
        raw['WL'] = raw['W'] / (raw['W'] + raw['L'])

    # Merge and average multi-team cities
    df = pd.merge(cities, raw, left_on=sport, right_on='team_key')
    df = df.groupby('Metropolitan area', as_index=False).agg(
        Population=('Population (2016 est.)[8]', 'first'),
        WL=('WL', 'mean')
    )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 1 — NHL  (expected: 28 metropolitan areas, r ≈ 0.0125)
# ─────────────────────────────────────────────────────────────────────────────

def nhl_correlation():
    df = build_sport_df('NHL', 'nhl.csv')          # no wl_col → computed from W/L

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q1: There should be 28 teams being analysed for NHL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 2 — NBA  (expected: 28 metropolitan areas, r ≈ -0.1764)
# ─────────────────────────────────────────────────────────────────────────────

def nba_correlation():
    df = build_sport_df('NBA', 'nba.csv', wl_col='W/L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q2: There should be 28 teams being analysed for NBA"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 3 — MLB  (expected: 26 metropolitan areas, r ≈ 0.1500)
# ─────────────────────────────────────────────────────────────────────────────

def mlb_correlation():
    df = build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, \
        "Q3: There should be 26 teams being analysed for MLB"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 4 — NFL  (expected: 29 metropolitan areas, r ≈ 0.0043)
# ─────────────────────────────────────────────────────────────────────────────

def nfl_correlation():
    df = build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, \
        "Q4: There should be 29 teams being analysed for NFL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 5 — Paired t-tests across all sport pairs
# ─────────────────────────────────────────────────────────────────────────────

def sports_team_performance():
    """
    For every pair of sports, run a paired t-test (ttest_rel) on the W-L%
    of cities that have teams in BOTH sports.  Returns a 4×4 DataFrame of
    p-values; the diagonal is NaN (same sport vs itself).

    Key assertions (checked by the grader):
        NBA vs NHL  p ≈ 0.022   (can reject null — they perform differently)
        MLB vs NFL  p ≈ 0.803   (cannot reject null)
    """
    # Reuse the same cleaned DataFrames from Q1-Q4
    leagues = {
        'NFL': build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
        'NBA': build_sport_df('NBA', 'nba.csv', wl_col='W/L%')[['Metropolitan area', 'WL']],
        'NHL': build_sport_df('NHL', 'nhl.csv')[['Metropolitan area', 'WL']],
        'MLB': build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
    }

    names = list(leagues.keys())
    p_values = pd.DataFrame(index=names, columns=names, dtype=float)

    for l1 in names:
        for l2 in names:
            if l1 == l2:
                p_values.loc[l1, l2] = np.nan
            else:
                # Only cities present in BOTH leagues
                merged = pd.merge(leagues[l1], leagues[l2], on='Metropolitan area')
                _, p = stats.ttest_rel(merged['WL_x'], merged['WL_y'])
                p_values.loc[l1, l2] = p

    assert abs(p_values.loc['NBA', 'NHL'] - 0.02) <= 1e-2, \
        "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc['MLB', 'NFL'] - 0.80) <= 1e-2, \
        "The MLB-NFL p-value should be around 0.80"

    return p_values


# ─────────────────────────────────────────────────────────────────────────────
# Quick sanity-check when run directly
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print(f"Q1 NHL  correlation : {nhl_correlation():.6f}")
    print(f"Q2 NBA  correlation : {nba_correlation():.6f}")
    print(f"Q3 MLB  correlation : {mlb_correlation():.6f}")
    print(f"Q4 NFL  correlation : {nfl_correlation():.6f}")
    print()
    print("Q5 p-values:")
    print(sports_team_performance().to_string())

## Question 4
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NFL** using **2018** data.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPERS  (used by all 5 questions)
# ─────────────────────────────────────────────────────────────────────────────

# Nicknames that are two words — must NOT be split when tokenising city entries
TWO_WORD_NICKNAMES = [
    'Red Sox', 'White Sox', 'Blue Jays', 'Red Wings',
    'Maple Leafs', 'Blue Jackets', 'Golden Knights', 'Trail Blazers'
]

def get_team_key(raw_name):
    """
    Extract a team's matchable nickname from its full name in the CSV.
    • Strips footnote markers (* + ) and NBA seed annotations (\\xa0(N)).
    • Returns last TWO words for two-word nicknames, otherwise last word.
    Examples:
        'Boston Red Sox*'            → 'Red Sox'
        'Portland Trail Blazers* (3)' → 'Trail Blazers'
        'Dallas Stars'               → 'Stars'
    """
    name = re.sub(r'[\*\+]', '', raw_name)
    name = re.sub(r'\xa0.*', '', name).strip()   # remove non-breaking space + seed
    words = name.split()
    if len(words) >= 2 and ' '.join(words[-2:]) in TWO_WORD_NICKNAMES:
        return ' '.join(words[-2:])
    return words[-1]


def tokenize_city_entry(raw):
    """
    Split a Wikipedia city cell (e.g. 'Cubs White Sox') into individual
    team nicknames while preserving two-word nicknames as single tokens.
    Returns a list of nickname strings (empty list for '—' / blank cells).
    """
    s = re.sub(r'\[.*?\]', '', raw).strip()   # strip footnotes like [note 3]
    if not s or s == '—':
        return []
    # Temporarily replace spaces inside two-word nicknames so split() won't break them
    protected = s
    for tw in sorted(TWO_WORD_NICKNAMES, key=len, reverse=True):
        protected = protected.replace(tw, tw.replace(' ', '\x00'))
    tokens = protected.split()
    return [t.replace('\x00', ' ') for t in tokens]


def load_cities(sport):
    """
    Load wikipedia_data.html, keep only Metropolitan area + population +
    the requested sport column, then explode multi-team cells into one row
    per team nickname.
    """
    cities = pd.read_html('assets/wikipedia_data.html')[1]
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities['Population (2016 est.)[8]'] = pd.to_numeric(
        cities['Population (2016 est.)[8]'], errors='coerce')
    cities[sport] = cities[sport].apply(tokenize_city_entry)
    cities = cities.explode(sport)
    # drop rows with no team (blank or dash)
    cities = cities[
        cities[sport].str.strip().ne('') & cities[sport].str.strip().ne('—')
    ].copy()
    return cities


def build_sport_df(sport, csv_file, wl_col=None):
    """
    Generic builder used by Q1-Q5.
    • Loads the sport CSV, keeps only 2018 rows, drops division-header rows.
    • Merges with city population data on team nickname.
    • Averages W-L% for cities with multiple teams.
    • Returns a DataFrame with columns: Metropolitan area | Population | WL
    """
    raw = pd.read_csv(f'assets/{csv_file}')
    cities = load_cities(sport)

    # Keep only 2018 and real team rows (filter out division-header rows)
    raw = raw[raw['year'] == 2018].copy()
    raw = raw[pd.to_numeric(raw['W'], errors='coerce').notna()].copy()

    # Build the matchable key
    raw['team_key'] = raw['team'].apply(get_team_key)

    # Win-loss percentage
    if wl_col:
        raw['WL'] = pd.to_numeric(raw[wl_col], errors='coerce')
    else:                                       # NHL has no pre-computed W-L%
        raw['W'] = raw['W'].astype(int)
        raw['L'] = raw['L'].astype(int)
        raw['WL'] = raw['W'] / (raw['W'] + raw['L'])

    # Merge and average multi-team cities
    df = pd.merge(cities, raw, left_on=sport, right_on='team_key')
    df = df.groupby('Metropolitan area', as_index=False).agg(
        Population=('Population (2016 est.)[8]', 'first'),
        WL=('WL', 'mean')
    )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 1 — NHL  (expected: 28 metropolitan areas, r ≈ 0.0125)
# ─────────────────────────────────────────────────────────────────────────────

def nhl_correlation():
    df = build_sport_df('NHL', 'nhl.csv')          # no wl_col → computed from W/L

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q1: There should be 28 teams being analysed for NHL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 2 — NBA  (expected: 28 metropolitan areas, r ≈ -0.1764)
# ─────────────────────────────────────────────────────────────────────────────

def nba_correlation():
    df = build_sport_df('NBA', 'nba.csv', wl_col='W/L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q2: There should be 28 teams being analysed for NBA"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 3 — MLB  (expected: 26 metropolitan areas, r ≈ 0.1500)
# ─────────────────────────────────────────────────────────────────────────────

def mlb_correlation():
    df = build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, \
        "Q3: There should be 26 teams being analysed for MLB"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 4 — NFL  (expected: 29 metropolitan areas, r ≈ 0.0043)
# ─────────────────────────────────────────────────────────────────────────────

def nfl_correlation():
    df = build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, \
        "Q4: There should be 29 teams being analysed for NFL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 5 — Paired t-tests across all sport pairs
# ─────────────────────────────────────────────────────────────────────────────

def sports_team_performance():
    """
    For every pair of sports, run a paired t-test (ttest_rel) on the W-L%
    of cities that have teams in BOTH sports.  Returns a 4×4 DataFrame of
    p-values; the diagonal is NaN (same sport vs itself).

    Key assertions (checked by the grader):
        NBA vs NHL  p ≈ 0.022   (can reject null — they perform differently)
        MLB vs NFL  p ≈ 0.803   (cannot reject null)
    """
    # Reuse the same cleaned DataFrames from Q1-Q4
    leagues = {
        'NFL': build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
        'NBA': build_sport_df('NBA', 'nba.csv', wl_col='W/L%')[['Metropolitan area', 'WL']],
        'NHL': build_sport_df('NHL', 'nhl.csv')[['Metropolitan area', 'WL']],
        'MLB': build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
    }

    names = list(leagues.keys())
    p_values = pd.DataFrame(index=names, columns=names, dtype=float)

    for l1 in names:
        for l2 in names:
            if l1 == l2:
                p_values.loc[l1, l2] = np.nan
            else:
                # Only cities present in BOTH leagues
                merged = pd.merge(leagues[l1], leagues[l2], on='Metropolitan area')
                _, p = stats.ttest_rel(merged['WL_x'], merged['WL_y'])
                p_values.loc[l1, l2] = p

    assert abs(p_values.loc['NBA', 'NHL'] - 0.02) <= 1e-2, \
        "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc['MLB', 'NFL'] - 0.80) <= 1e-2, \
        "The MLB-NFL p-value should be around 0.80"

    return p_values


# ─────────────────────────────────────────────────────────────────────────────
# Quick sanity-check when run directly
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print(f"Q1 NHL  correlation : {nhl_correlation():.6f}")
    print(f"Q2 NBA  correlation : {nba_correlation():.6f}")
    print(f"Q3 MLB  correlation : {mlb_correlation():.6f}")
    print(f"Q4 NFL  correlation : {nfl_correlation():.6f}")
    print()
    print("Q5 p-values:")
    print(sports_team_performance().to_string())

## Question 5
In this question I would like you to explore the hypothesis that **given that an area has two sports teams in different sports, those teams will perform the same within their respective sports**. How I would like to see this explored is with a series of paired t-tests (so use [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) between all pairs of sports. Are there any sports where we can reject the null hypothesis? Again, average values where a sport has multiple teams in one region. Remember, you will only be including, for each sport, cities which have teams engaged in that sport, drop others as appropriate. This question is worth 20% of the grade for this assignment.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPERS  (used by all 5 questions)
# ─────────────────────────────────────────────────────────────────────────────

# Nicknames that are two words — must NOT be split when tokenising city entries
TWO_WORD_NICKNAMES = [
    'Red Sox', 'White Sox', 'Blue Jays', 'Red Wings',
    'Maple Leafs', 'Blue Jackets', 'Golden Knights', 'Trail Blazers'
]

def get_team_key(raw_name):
    """
    Extract a team's matchable nickname from its full name in the CSV.
    • Strips footnote markers (* + ) and NBA seed annotations (\\xa0(N)).
    • Returns last TWO words for two-word nicknames, otherwise last word.
    Examples:
        'Boston Red Sox*'            → 'Red Sox'
        'Portland Trail Blazers* (3)' → 'Trail Blazers'
        'Dallas Stars'               → 'Stars'
    """
    name = re.sub(r'[\*\+]', '', raw_name)
    name = re.sub(r'\xa0.*', '', name).strip()   # remove non-breaking space + seed
    words = name.split()
    if len(words) >= 2 and ' '.join(words[-2:]) in TWO_WORD_NICKNAMES:
        return ' '.join(words[-2:])
    return words[-1]


def tokenize_city_entry(raw):
    """
    Split a Wikipedia city cell (e.g. 'Cubs White Sox') into individual
    team nicknames while preserving two-word nicknames as single tokens.
    Returns a list of nickname strings (empty list for '—' / blank cells).
    """
    s = re.sub(r'\[.*?\]', '', raw).strip()   # strip footnotes like [note 3]
    if not s or s == '—':
        return []
    # Temporarily replace spaces inside two-word nicknames so split() won't break them
    protected = s
    for tw in sorted(TWO_WORD_NICKNAMES, key=len, reverse=True):
        protected = protected.replace(tw, tw.replace(' ', '\x00'))
    tokens = protected.split()
    return [t.replace('\x00', ' ') for t in tokens]


def load_cities(sport):
    """
    Load wikipedia_data.html, keep only Metropolitan area + population +
    the requested sport column, then explode multi-team cells into one row
    per team nickname.
    """
    cities = pd.read_html('assets/wikipedia_data.html')[1]
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities['Population (2016 est.)[8]'] = pd.to_numeric(
        cities['Population (2016 est.)[8]'], errors='coerce')
    cities[sport] = cities[sport].apply(tokenize_city_entry)
    cities = cities.explode(sport)
    # drop rows with no team (blank or dash)
    cities = cities[
        cities[sport].str.strip().ne('') & cities[sport].str.strip().ne('—')
    ].copy()
    return cities


def build_sport_df(sport, csv_file, wl_col=None):
    """
    Generic builder used by Q1-Q5.
    • Loads the sport CSV, keeps only 2018 rows, drops division-header rows.
    • Merges with city population data on team nickname.
    • Averages W-L% for cities with multiple teams.
    • Returns a DataFrame with columns: Metropolitan area | Population | WL
    """
    raw = pd.read_csv(f'assets/{csv_file}')
    cities = load_cities(sport)

    # Keep only 2018 and real team rows (filter out division-header rows)
    raw = raw[raw['year'] == 2018].copy()
    raw = raw[pd.to_numeric(raw['W'], errors='coerce').notna()].copy()

    # Build the matchable key
    raw['team_key'] = raw['team'].apply(get_team_key)

    # Win-loss percentage
    if wl_col:
        raw['WL'] = pd.to_numeric(raw[wl_col], errors='coerce')
    else:                                       # NHL has no pre-computed W-L%
        raw['W'] = raw['W'].astype(int)
        raw['L'] = raw['L'].astype(int)
        raw['WL'] = raw['W'] / (raw['W'] + raw['L'])

    # Merge and average multi-team cities
    df = pd.merge(cities, raw, left_on=sport, right_on='team_key')
    df = df.groupby('Metropolitan area', as_index=False).agg(
        Population=('Population (2016 est.)[8]', 'first'),
        WL=('WL', 'mean')
    )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 1 — NHL  (expected: 28 metropolitan areas, r ≈ 0.0125)
# ─────────────────────────────────────────────────────────────────────────────

def nhl_correlation():
    df = build_sport_df('NHL', 'nhl.csv')          # no wl_col → computed from W/L

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q1: There should be 28 teams being analysed for NHL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 2 — NBA  (expected: 28 metropolitan areas, r ≈ -0.1764)
# ─────────────────────────────────────────────────────────────────────────────

def nba_correlation():
    df = build_sport_df('NBA', 'nba.csv', wl_col='W/L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, \
        "Q2: There should be 28 teams being analysed for NBA"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 3 — MLB  (expected: 26 metropolitan areas, r ≈ 0.1500)
# ─────────────────────────────────────────────────────────────────────────────

def mlb_correlation():
    df = build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, \
        "Q3: There should be 26 teams being analysed for MLB"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 4 — NFL  (expected: 29 metropolitan areas, r ≈ 0.0043)
# ─────────────────────────────────────────────────────────────────────────────

def nfl_correlation():
    df = build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')

    population_by_region = df['Population']
    win_loss_by_region   = df['WL']

    assert len(population_by_region) == len(win_loss_by_region), \
        "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, \
        "Q4: There should be 29 teams being analysed for NFL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]


# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 5 — Paired t-tests across all sport pairs
# ─────────────────────────────────────────────────────────────────────────────

def sports_team_performance():
    """
    For every pair of sports, run a paired t-test (ttest_rel) on the W-L%
    of cities that have teams in BOTH sports.  Returns a 4×4 DataFrame of
    p-values; the diagonal is NaN (same sport vs itself).

    Key assertions (checked by the grader):
        NBA vs NHL  p ≈ 0.022   (can reject null — they perform differently)
        MLB vs NFL  p ≈ 0.803   (cannot reject null)
    """
    # Reuse the same cleaned DataFrames from Q1-Q4
    leagues = {
        'NFL': build_sport_df('NFL', 'nfl.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
        'NBA': build_sport_df('NBA', 'nba.csv', wl_col='W/L%')[['Metropolitan area', 'WL']],
        'NHL': build_sport_df('NHL', 'nhl.csv')[['Metropolitan area', 'WL']],
        'MLB': build_sport_df('MLB', 'mlb.csv', wl_col='W-L%')[['Metropolitan area', 'WL']],
    }

    names = list(leagues.keys())
    p_values = pd.DataFrame(index=names, columns=names, dtype=float)

    for l1 in names:
        for l2 in names:
            if l1 == l2:
                p_values.loc[l1, l2] = np.nan
            else:
                # Only cities present in BOTH leagues
                merged = pd.merge(leagues[l1], leagues[l2], on='Metropolitan area')
                _, p = stats.ttest_rel(merged['WL_x'], merged['WL_y'])
                p_values.loc[l1, l2] = p

    assert abs(p_values.loc['NBA', 'NHL'] - 0.02) <= 1e-2, \
        "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc['MLB', 'NFL'] - 0.80) <= 1e-2, \
        "The MLB-NFL p-value should be around 0.80"

    return p_values


# ─────────────────────────────────────────────────────────────────────────────
# Quick sanity-check when run directly
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print(f"Q1 NHL  correlation : {nhl_correlation():.6f}")
    print(f"Q2 NBA  correlation : {nba_correlation():.6f}")
    print(f"Q3 MLB  correlation : {mlb_correlation():.6f}")
    print(f"Q4 NFL  correlation : {nfl_correlation():.6f}")
    print()
    print("Q5 p-values:")
    print(sports_team_performance().to_string())